# ML-KEM-512 `cmov` Capture on Husky

Este notebook queda ajustado al firmware actual de este directorio con `SS_VER_2_1`.

Puntos importantes verificados en el repo:
- El `makefile` local ahora fuerza `SS_VER_2_1` por defecto.
- El firmware usa `cmd = 0x01` con subcomandos para cargar `sk`, `pk`, `ct` y `ss`, y `scmd = 0x40` para lanzar la decapsulacion.
- El trigger en firmware rodea solo `PQCLEAN_MLKEM512_CLEAN_cmov(...)`.
- La respuesta calculada vuelve por `r` con 32 bytes y luego llega el `ack` `e`.

Este notebook, por ahora, toma los vectores desde los arreglos del `.c` y luego los sube por chunks al target. Si luego decides dejar esos arreglos en ceros en firmware, la fuente de vectores debera moverse a este notebook o a un archivo externo (`npz`, `json`, `bin`).

Limitacion importante:
- Como repites el mismo `sk` y el mismo `ct`, este notebook no hace un CPA clasico entre trazas.
- En su lugar deja una rutina automatica de exploracion para `cmov` basada en ventanas por byte del bucle de 32 bytes.

In [1]:
from pathlib import Path
import json
import re
import time

try:
    import chipwhisperer as cw
except ModuleNotFoundError:
    cw = None
    print('Warning: chipwhisperer no esta instalado en este kernel; las celdas de hardware no funcionaran.')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from tqdm.notebook import trange

WORKDIR = Path.cwd()
FIRMWARE_C = Path('/opt/tools/chipwhisperer/firmware/mcu/simpleserial-ml-kem-512/simpleserial-ml-kem-512.c')

hex_candidates = sorted(WORKDIR.glob('simpleserial-ml-kem-512-*.hex'))
if not hex_candidates:
    raise FileNotFoundError(f'No .hex found in {WORKDIR}. Build firmware first.')
FIRMWARE_HEX = hex_candidates[0]

OUTPUT_DIR = WORKDIR / 'cmov_capture_output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PLATFORM = 'CWHUSKY'
SS_VER = 'SS_VER_2_1'
N_TRACES = 100
ADC_SAMPLES = 3000
ADC_OFFSET = 0
GAIN_DB = 18
CAPTURE_TIMEOUT = 5
PREVIEW_COUNT = 20
ACK_TIMEOUT_MS = 2000
READ_TIMEOUT_MS = 5000

CMOV_BYTES = 32

SS_CMD = 0x01
SCMD_SK_BASE = 0x00
SCMD_PK_BASE = 0x10
SCMD_CT_BASE = 0x20
SCMD_SS = 0x30
SCMD_DECAP = 0x40

SK_CHUNK_LEN = 204
PK_CHUNK_LEN = 200
CT_CHUNK_LEN = 192
SK_CHUNKS = 8
PK_CHUNKS = 4
CT_CHUNKS = 4

LED1 = bytes([16])
LED2 = bytes([15])
LED3 = bytes([14])

DELTA_SS = 'delta_ss'
SS = 'ss'
SS2 = 'ss2'
SK_FIRST32 = 'sk_first32'
CT_FIRST32 = 'ct_first32'
DEFAULT_MODEL_NAME = DELTA_SS
MODEL_NAME = DEFAULT_MODEL_NAME

print(f'Firmware source: {FIRMWARE_C}')
print(f'Firmware hex: {FIRMWARE_HEX}')
print(f'Output directory: {OUTPUT_DIR}')
print(f'Traces planned: {N_TRACES}')
print(f'ADC samples: {ADC_SAMPLES}')

Firmware source: /opt/tools/chipwhisperer/firmware/mcu/simpleserial-ml-kem-512/simpleserial-ml-kem-512.c
Firmware hex: /opt/tools/chipwhisperer/firmware/mcu/simpleserial-ml-kem-512/simpleserial-ml-kem-512-CWHUSKY.hex
Output directory: /opt/tools/chipwhisperer/firmware/mcu/simpleserial-ml-kem-512/cmov_capture_output
Traces planned: 100
ADC samples: 3000


In [2]:
def parse_c_byte_array(source_text: str, name: str) -> np.ndarray:

    pattern = rf'uint8_t\s+{re.escape(name)}\[[^\]]+\]\s*=\s*\{{(.*?)\}};'

    match = re.search(pattern, source_text, re.S)

    if match is None:

        raise ValueError(f'Array {name!r} not found in {FIRMWARE_C}')



    body = match.group(1)

    values = []

    for token in body.replace('\n', ' ').split(','):

        token = token.strip()

        if not token:

            continue

        values.append(int(token, 16))

    return np.array(values, dtype=np.uint8)





def hex_to_u8(hex_str: str, name: str) -> np.ndarray:

    if len(hex_str) % 2 != 0:

        raise ValueError(f'{name}: hex len impar')

    return np.frombuffer(bytes.fromhex(hex_str), dtype=np.uint8).copy()





def expand_zero_initializer(values: np.ndarray, expected_len: int, name: str) -> np.ndarray:

    values = np.asarray(values, dtype=np.uint8)

    if len(values) == expected_len:

        return values

    # Soporta inicializadores C compactos tipo {0} en arrays grandes.

    if len(values) == 1 and int(values[0]) == 0:

        return np.zeros(expected_len, dtype=np.uint8)

    raise ValueError(f'{name} len={len(values)} no coincide con expected_len={expected_len}')





def hw_u8(values: np.ndarray) -> np.ndarray:

    values = np.asarray(values, dtype=np.uint8)

    return np.unpackbits(values[:, None], axis=1).sum(axis=1)





def bit_column(values: np.ndarray, bit_index: int) -> np.ndarray:

    values = np.asarray(values, dtype=np.uint8)

    return ((values >> bit_index) & 1).astype(np.uint8)





# Backup rescatado del firmware antes de limpiar vectores fijos.

RESCUED_VECTORS_HEX = {

    'pk': '48815885b5662dd45d777c1cc3352d8cfa8f634602bc78524662c23c5a3b84a4274a490e0349863685af9b566bad31a878b7c26d6c8f3fa5aef744b1abf95814bbb13fca4295e90beda58b058795643539f04332dc614a6f57c26f75a16d4073b5622b40c14ba9fb8afeec38cd1419d6356a723abb1a9136b788ac2b2852c0ac2871d9606c3a1d8840ca7c3a106a3757fa4b2973b93afef6948d836960263696d81708d0015eaa8f92e3693996b64361bc8c6559a6262258855e69327edb1aa02470ab6bba7908c9a8bc46553b582c18500c2070a758f27b6278a7f0439124511846782b7cf26976034a97492b47930087281957c02ebf91942819858af03b3c86877e601d37d019d3775c3c99310c590abc22c20d664e87393a7629cd189a51faacbd2a6486397cae2b1299f2623fae817d56aabcd0c144df83a6356111e0129b3be6148a450d2130b06e4b3b00c69de5fba5ede8256fdc7e9de629e05a6320c84629e93bfe22b1438b837fdb4d0cb67ebcc81443f12c17614ffae6a9d1babadda88679bb49b53a9ec0b63c2c499bf5288c6c067a5ca852c515bb8266b971d07fbb1a917fc66a8c38a0f1578d01a551e1ca6b1b9278b13793b892c002bc995f81b5d2e3c03bb66ed1091922c38812faa1f7c87fa5321bc7637965b8bbbe16170651c065cbacb24018b25ac9360b98a7175abdc0221f070dd8383a150074703c2b87d397f4c337fe5951b515b7f2ca783d1a975b8b2ab75a3cf1795ec4249d3b705395e090c7aaa5b2274a4de83315c39f5ad5a99db151cbd98a5508a6f74675632c8fda876165880b88a7aa309645bd446ae0a9977dc125dab972d99a3e0ad11c33246e56c09be3c5ad1164178f161b88b76ef5680dd6e54cf7781f2a244366b413fb5b7a88861fa635894a93a931bb792ab8516b5c5adc993a4636215fc861676b44ca372cc1524cb987727fa625e4617d8a43026d40c7d5f28ea32505580845fe9ac71444b7c44c2991660069f0c46627b5a9d60ce3b63a245aa5b9f330769c4979675ed14800ddd44e69d040b1d299d4897235020aeef2aaef0124ed3c1c153fe6c6426db948d42cf5cced5cf548a40160f1878a9b0f8019c3ae912ad54a',

    'sk': '78d38c15d4ca32a353a7dcb510982b9c55628a5c7dd18a1123e4733e223ab6b57b77a789604b424fc988f3c62089d17238ac9b0eb8af2bd281f3dc727782c3fd4587028609370162cdcb2329845b130a5915c0596b59a587672ad1a2bd085a9482233ed1e14d5492583a05468236ba30086dcb311e6df8ad92f368c6860ac1c819f60905634b6be32218a2f11a4757291327341784446869988757c13455473a2884b275211682c6bcf922cbd81732b1c71979c39d597ca712347060a24069a6fdf7c95d100932c8ad523a32fa7b6a361990cb673842b0142bc29ad10b846957388e59b1da5cbfb35ccd0617888060708a7597ae042c2faab4a1809a522c52a8f1524323969c10772e8479bc66b8efb026d8d9581d130f0a681ad2b29054f56659f2b338b5ae5a193339d64f93b2cfefc6090865bb7353a8082027b643b87ab22a802094154ac1cae3701445b3c5120c11f6253d6293644a55129607c43023090475c9eb785571ce09e4132c08a26be4c054a8804423a30bfbc0ce5896ac9025ded48e871b235ea5492af37108586dcdf388621989696837b9b58577a1bae1079733a09f1c2979e4994545268bf5427b4fcbb3e693045d32425b69880e6500d46672bea8ce7e36627c680d6ec53a70e094f3743d136bc67344a4066251113b6e3795876827becb145f53ec54dbfc756f9a0706d03ca8038d4471a17b8934f920ad9fc775c9ac9d7e888c79eb24a0771b1eebb8f60658d737b519ec36a915c0a2a77522f10c0492439c774d1628cc1660af4d1c1db6b72195103700103ac0f5ca2461c479da2ff018511fb57c701525c50ab5e19915aa77228c9245e139ad21c7b8fb7335e2789fed32662f21515de14710501dbf991de14417fed32f961c5d6db7770e676baeb67b485c9a4d35a14a0ac876912e7d24c32247c0ab704ec3ab468fc10bb963685d26862b0b65b5f8ab51841af4748e193712f2cb82e476673466b35e56269cc93c5f3a8563343a8a9a9ba17ac6860860df0916da6bae8352587c9256aa77c86e9abbed3765bb793dff17784e25bc786b35a03798230924c8247a48815885b5662dd45d777c1cc3352d8cfa8f634602bc78524662c23c5a3b84a4274a490e0349863685af9b566bad31a878b7c26d6c8f3fa5aef744b1abf95814bbb13fca4295e90beda58b058795643539f04332dc614a6f57c26f75a16d4073b5622b40c14ba9fb8afeec38cd1419d6356a723abb1a9136b788ac2b2852c0ac2871d9606c3a1d8840ca7c3a106a3757fa4b2973b93afef6948d836960263696d81708d0015eaa8f92e3693996b64361bc8c6559a6262258855e69327edb1aa02470ab6bba7908c9a8bc46553b582c18500c2070a758f27b6278a7f0439124511846782b7cf26976034a97492b47930087281957c02ebf91942819858af03b3c86877e601d37d019d3775c3c99310c590abc22c20d664e87393a7629cd189a51faacbd2a6486397cae2b1299f2623fae817d56aabcd0c144df83a6356111e0129b3be6148a450d2130b06e4b3b00c69de5fba5ede8256fdc7e9de629e05a6320c84629e93bfe22b1438b837fdb4d0cb67ebcc81443f12c17614ffae6a9d1babadda88679bb49b53a9ec0b63c2c499bf5288c6c067a5ca852c515bb8266b971d07fbb1a917fc66a8c38a0f1578d01a551e1ca6b1b9278b13793b892c002bc995f81b5d2e3c03bb66ed1091922c38812faa1f7c87fa5321bc7637965b8bbbe16170651c065cbacb24018b25ac9360b98a7175abdc0221f070dd8383a150074703c2b87d397f4c337fe5951b515b7f2ca783d1a975b8b2ab75a3cf1795ec4249d3b705395e090c7aaa5b2274a4de83315c39f5ad5a99db151cbd98a5508a6f74675632c8fda876165880b88a7aa309645bd446ae0a9977dc125dab972d99a3e0ad11c33246e56c09be3c5ad1164178f161b88b76ef5680dd6e54cf7781f2a244366b413fb5b7a88861fa635894a93a931bb792ab8516b5c5adc993a4636215fc861676b44ca372cc1524cb987727fa625e4617d8a43026d40c7d5f28ea32505580845fe9ac71444b7c44c2991660069f0c46627b5a9d60ce3b63a245aa5b9f330769c4979675ed14800ddd44e69d040b1d299d4897235020aeef2aaef0124ed3c1c153fe6c6426db948d42cf5cced5cf548a40160f1878a9b0f8019c3ae912ad54a733d6fb4d92d2c896d3f1d168900e535299f8ef4d8305399e29f0f35f6571b080f32a1dd56979171abcf86302b6c9a4180de4e8aece98d926049d1e9f59292fd',

    'ct': '463fc09919479cd668df9701c76569bcaac43b7ca541aaa072b102359a1e13c56403a39ccb50bcd5ab7ff96eae2334621652b9d7c9db3457606b8606a94efdb9e572654431b6f23ac7e4aab99b1578771e8a30b9bbb67871b8dfe66baebe3063f8d3cb9c6c9d5ca128640e0a47b27a5e96a19e6ad206af5942d2f2aa35200fa25418d7d223514ba4ee3a9fd8b57e3d168e35d1918c66b31c4429af054fbd2b9d8d03c60986b8c6802233a575f5a516cfda0abee0ce896c7b9c75afd8ac0794145e1714050cf73795c4c4529d34e23ae26bc01dee8bde60a5c4381cea82cfb95997ba8ae03947b5e7288e587377c4542334baf3f2cae5501b8616153b01053fe2c53fe91abedcf1d8455417e96ecbf9f96f6336d79f245477bfe862b41beea7d05c2f84e42bc0e590d157c080af17f6770f28cc1dd3cf7034a1fbe7a392844a897d8eb72e6021fa3ca4b0adbc6a46b4a317143bac577ead293cbc9e10d3c9d9849be450bfb51b4d977b91b06851cd4b7ec04c7676b7d1e40d3acebfe8557658f6dd778333e2e1f0ce40dd951f827174fbe9f0533ca73260ce819f168fb131626d9018c573565484fa4527889b8c68ab58756c30a52b408166b108ee0e171423a22ad8aa59a8fad1992424b7977023c2eac8da38b9cb82d5b874a9ebaaaf021948c4c5db574cc0f53cbcfd45d40b8af88a2e831e547f6367f7ec5077429e838f98af1f8ca2c90a210601265aedec7e1991c72c377bf1b257b3ffd5bc8639ebca4ece52c34e069c9edb7dd98f44cee46cdaa0702a2f9490b0f561a57c1cfffa50b975d526f2918082d6a6dfe3094071fee4ffccb76c150d5e55f65a87d96dceeb06d3e8f5ca16fe968645ee6efdd34a505374fd13c9d60ff7d572c7cc4c537cc253baa177f3fb6219ac27318b145ea17d81e5050b4c56df01d34ebfb8de46639945bd5de8eb07a97b3f95a7218293b8f9931f1566810e73d0abea8bed3bc85af63516346e1894b33c4b767764166a5d5ed937822e314efa91924ce729074fa53871257e615c2bbb3714eb91dd1fc896b56de64645a3678c1e7aafe33918913108ef',

    'ss': '17e15971054cd3ac2b416ef418d774ede04625d805102e246bed45468007530e',

    'ss2': 'fb7385bf2239cb99565b3dd8faebe33c6feec265464bd8c934a02e7ffbaea3f0',

}



USE_RESCUED_VECTORS = True



expected_pk_len = PK_CHUNK_LEN * PK_CHUNKS

expected_sk_len = SK_CHUNK_LEN * SK_CHUNKS

expected_ct_len = CT_CHUNK_LEN * CT_CHUNKS

expected_ss_len = CMOV_BYTES



if USE_RESCUED_VECTORS:

    pk = expand_zero_initializer(hex_to_u8(RESCUED_VECTORS_HEX['pk'], 'pk'), expected_pk_len, 'pk')

    sk = expand_zero_initializer(hex_to_u8(RESCUED_VECTORS_HEX['sk'], 'sk'), expected_sk_len, 'sk')

    ct = expand_zero_initializer(hex_to_u8(RESCUED_VECTORS_HEX['ct'], 'ct'), expected_ct_len, 'ct')

    ss = expand_zero_initializer(hex_to_u8(RESCUED_VECTORS_HEX['ss'], 'ss'), expected_ss_len, 'ss')

    ss2 = expand_zero_initializer(hex_to_u8(RESCUED_VECTORS_HEX['ss2'], 'ss2'), expected_ss_len, 'ss2')

else:

    source_text = FIRMWARE_C.read_text()

    pk = expand_zero_initializer(parse_c_byte_array(source_text, 'pk'), expected_pk_len, 'pk')

    sk = expand_zero_initializer(parse_c_byte_array(source_text, 'sk'), expected_sk_len, 'sk')

    ct = expand_zero_initializer(parse_c_byte_array(source_text, 'ct'), expected_ct_len, 'ct')

    ss = expand_zero_initializer(parse_c_byte_array(source_text, 'ss'), expected_ss_len, 'ss')

    ss2 = expand_zero_initializer(parse_c_byte_array(source_text, 'ss2'), expected_ss_len, 'ss2')



reference_vectors = {

    'pk': pk,

    'sk': sk,

    'ct': ct,

    'ss': ss,

    'ss2': ss2,

    'delta_ss': np.bitwise_xor(ss, ss2),

    'sk_first32': sk[:CMOV_BYTES],

    'ct_first32': ct[:CMOV_BYTES],

}



for name, values in reference_vectors.items():

    print(f'{name:10s} len={len(values):4d} first8={[hex(int(v)) for v in values[:8]]}')



reference_df = pd.DataFrame({

    'byte_index': np.arange(CMOV_BYTES),

    'ss': ss[:CMOV_BYTES],

    'ss2': ss2[:CMOV_BYTES],

    'delta_ss': np.bitwise_xor(ss[:CMOV_BYTES], ss2[:CMOV_BYTES]),

    'sk_first32': sk[:CMOV_BYTES],

    'ct_first32': ct[:CMOV_BYTES],

})

reference_df['ss_hw'] = hw_u8(reference_df['ss'].to_numpy(dtype=np.uint8))

reference_df['ss2_hw'] = hw_u8(reference_df['ss2'].to_numpy(dtype=np.uint8))

reference_df['delta_hw'] = hw_u8(reference_df['delta_ss'].to_numpy(dtype=np.uint8))

reference_df['sk_hw'] = hw_u8(reference_df['sk_first32'].to_numpy(dtype=np.uint8))

reference_df.head()

pk         len= 800 first8=['0x48', '0x81', '0x58', '0x85', '0xb5', '0x66', '0x2d', '0xd4']
sk         len=1632 first8=['0x78', '0xd3', '0x8c', '0x15', '0xd4', '0xca', '0x32', '0xa3']
ct         len= 768 first8=['0x46', '0x3f', '0xc0', '0x99', '0x19', '0x47', '0x9c', '0xd6']
ss         len=  32 first8=['0x17', '0xe1', '0x59', '0x71', '0x5', '0x4c', '0xd3', '0xac']
ss2        len=  32 first8=['0xfb', '0x73', '0x85', '0xbf', '0x22', '0x39', '0xcb', '0x99']
delta_ss   len=  32 first8=['0xec', '0x92', '0xdc', '0xce', '0x27', '0x75', '0x18', '0x35']
sk_first32 len=  32 first8=['0x78', '0xd3', '0x8c', '0x15', '0xd4', '0xca', '0x32', '0xa3']
ct_first32 len=  32 first8=['0x46', '0x3f', '0xc0', '0x99', '0x19', '0x47', '0x9c', '0xd6']


,byte_index,ss,ss2,delta_ss,sk_first32,ct_first32,ss_hw,ss2_hw,delta_hw,sk_hw
0,0,23,251,236,120,70,4,7,5,4
1,1,225,115,146,211,63,4,5,3,5
2,2,89,133,220,140,192,4,3,5,3
3,3,113,191,206,21,153,4,7,5,3
4,4,5,34,39,212,25,2,2,4,4


In [3]:
# Herramientas para probar vectores personalizados sin editar firmware.

def _as_u8(arr, expected_len: int, name: str) -> np.ndarray:
    out = np.asarray(arr, dtype=np.uint8).copy()
    if len(out) != expected_len:
        raise ValueError(f'{name} len={len(out)} esperado={expected_len}')
    return out


def set_test_vectors(*, sk_new=None, pk_new=None, ct_new=None, ss_new=None, ss2_new=None):
    """Reemplaza uno o varios vectores globales usados por upload_reference_vectors()."""
    global sk, pk, ct, ss, ss2, reference_vectors

    if sk_new is not None:
        sk = _as_u8(sk_new, len(sk), 'sk')
    if pk_new is not None:
        pk = _as_u8(pk_new, len(pk), 'pk')
    if ct_new is not None:
        ct = _as_u8(ct_new, len(ct), 'ct')
    if ss_new is not None:
        ss = _as_u8(ss_new, len(ss), 'ss')
    if ss2_new is not None:
        ss2 = _as_u8(ss2_new, len(ss2), 'ss2')

    reference_vectors = {
        'pk': pk,
        'sk': sk,
        'ct': ct,
        'ss': ss,
        'ss2': ss2,
        'delta_ss': np.bitwise_xor(ss, ss2),
        'sk_first32': sk[:CMOV_BYTES],
        'ct_first32': ct[:CMOV_BYTES],
    }

    print('Vectores activos:')
    print(f"  sk first8  = {[hex(int(v)) for v in sk[:8]]}")
    print(f"  pk first8  = {[hex(int(v)) for v in pk[:8]]}")
    print(f"  ct first8  = {[hex(int(v)) for v in ct[:8]]}")
    print(f"  ss first8  = {[hex(int(v)) for v in ss[:8]]}")
    print(f"  ss2 first8 = {[hex(int(v)) for v in ss2[:8]]}")


def make_known_sk(pattern: str = 'incremental') -> np.ndarray:
    """Genera una SK conocida y reproducible para pruebas."""
    n = len(sk)
    if pattern == 'incremental':
        return (np.arange(n, dtype=np.uint16) & 0xFF).astype(np.uint8)
    if pattern == 'all_00':
        return np.zeros(n, dtype=np.uint8)
    if pattern == 'all_ff':
        return np.full(n, 0xFF, dtype=np.uint8)
    raise ValueError(f'pattern no soportado: {pattern}')


def derive_vectors_from_sk(sk_base: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Construye pk/ct/ss/ss2 a partir de sk base (mismo contenido, distinto largo)."""
    sk_base = _as_u8(sk_base, len(sk), 'sk_base')
    pk_new = sk_base[:len(pk)].copy()
    ct_new = sk_base[:len(ct)].copy()
    ss_new = sk_base[:len(ss)].copy()
    ss2_new = ss_new.copy()
    return sk_base, pk_new, ct_new, ss_new, ss2_new


def apply_known_bundle_from_sk(pattern: str = 'incremental', upload_now: bool = False) -> None:
    """
    Fija sk conocida y fuerza pk/ct/ss/ss2 con bytes derivados de esa misma sk.
    Si upload_now=True y upload_reference_vectors existe, sube todo por chunks al target.
    """
    sk_new, pk_new, ct_new, ss_new, ss2_new = derive_vectors_from_sk(make_known_sk(pattern))
    set_test_vectors(sk_new=sk_new, pk_new=pk_new, ct_new=ct_new, ss_new=ss_new, ss2_new=ss2_new)

    if upload_now:
        if 'upload_reference_vectors' not in globals():
            print('upload_now=True pero upload_reference_vectors() aun no existe. Ejecuta la celda de Prueba 1/2 primero.')
        else:
            upload_reference_vectors()
            print('Bundle conocido enviado por chunks.')


# Uso recomendado:
# 1) Ejecuta esta celda.
# 2) Luego ejecuta la celda de Prueba 1/2 (para definir upload_reference_vectors).
# 3) Finalmente ejecuta:
# apply_known_bundle_from_sk(pattern='incremental', upload_now=True)
#
# Nota: con este bundle, ss es derivada de sk y NO necesariamente coincide con ss_result real de decap,
# por lo que ACK=1 puede ser esperable si EXPECT_SS_MATCH=False.

## Conexion y configuracion

La configuracion arranca conservadora para una primera corrida sobre Husky:
- `ADC_SAMPLES = 3000`
- `ADC_OFFSET = 0`
- `GAIN_DB = 18`

Si el trigger queda bien centrado y ves demasiado margen libre, reduce `ADC_SAMPLES`.

In [4]:
try:
    if not scope.connectStatus:
        scope.con()
except NameError:
    scope = cw.scope()

try:
    target = cw.target(scope, cw.targets.SimpleSerial2)
except IOError:
    print('INFO: Caught exception on reconnecting to target - attempting to reconnect to scope first.')
    print('INFO: This is a work-around when USB has died without Python knowing. Ignore errors above this line.')
    scope = cw.scope()
    target = cw.target(scope, cw.targets.SimpleSerial2)

print('INFO: Found ChipWhisperer')

if 'STM' in PLATFORM or PLATFORM == 'CWLITEARM' or PLATFORM == 'CWNANO':
    prog = cw.programmers.STM32FProgrammer
elif PLATFORM == 'CW303' or PLATFORM == 'CWLITEXMEGA':
    prog = cw.programmers.XMEGAProgrammer
elif 'neorv32' in PLATFORM.lower():
    prog = cw.programmers.NEORV32Programmer
elif PLATFORM == 'CW308_SAM4S' or PLATFORM == 'CWHUSKY':
    prog = cw.programmers.SAM4SProgrammer
else:
    prog = None

time.sleep(0.05)
scope.default_setup()

if PLATFORM == 'CW308_SAM4S' or PLATFORM == 'CWHUSKY':
    scope.io.target_pwr = 0
    time.sleep(0.2)
    scope.io.target_pwr = 1
    time.sleep(0.2)

def reset_target(scope):
    if PLATFORM == 'CW303' or PLATFORM == 'CWLITEXMEGA':
        scope.io.pdic = 'low'
        time.sleep(0.1)
        scope.io.pdic = 'high_z'
        time.sleep(0.1)
    elif 'neorv32' in PLATFORM.lower():
        raise IOError('Default iCE40 neorv32 build does not have external reset - reprogram device to reset')
    elif PLATFORM == 'CW308_SAM4S' or PLATFORM == 'CWHUSKY':
        scope.io.nrst = 'low'
        time.sleep(0.25)
        scope.io.nrst = 'high_z'
        time.sleep(0.25)
    else:
        scope.io.nrst = 'low'
        time.sleep(0.05)
        scope.io.nrst = 'high_z'
        time.sleep(0.05)

scope.gain.db = GAIN_DB
scope.adc.offset = ADC_OFFSET
scope.adc.samples = ADC_SAMPLES
scope.adc.timeout = CAPTURE_TIMEOUT

cw.program_target(scope, prog, str(FIRMWARE_HEX))
reset_target(scope)
target.flush()

print(f'SS_VER notebook = {SS_VER}')
print(f'target class     = {type(target).__name__}')
print(f'gain.db         = {scope.gain.db}')
print(f'adc.offset      = {scope.adc.offset}')
print(f'adc.samples     = {scope.adc.samples}')
print(f'adc.timeout     = {scope.adc.timeout}')
print(f'firmware hex    = {FIRMWARE_HEX}')

INFO: Found ChipWhisperer
scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 327828                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.clock.adc_rate                     changed from 0.0                       to 29454545.454545453       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2                             changed from None                      to clkgen             

In [5]:
def ack_to_int(ack) -> int | None:
    if ack is None:
        return None
    if isinstance(ack, int):
        return ack
    if isinstance(ack, (bytes, bytearray)):
        return ack[0] if len(ack) > 0 else 0
    return int(ack)


def send_led_command(command: str, led_value: bytes) -> None:
    # En SSv2.1, enviar LED como comando + subcomando para recibir ACK consistente.
    target.send_cmd(ord(command), 0x00, bytearray(led_value))
    ack_raw = target.simpleserial_wait_ack(timeout=ACK_TIMEOUT_MS)
    ack = ack_to_int(ack_raw)
    if ack is None:
        raise RuntimeError(f'No ACK for LED command {command!r}')
    if ack != 0:
        raise RuntimeError(f'LED command {command!r} returned non-zero ACK: raw={ack_raw!r}, int={ack}')
    time.sleep(0.15)


def send_v21_cmd(scmd: int, payload: bytes | bytearray = b'') -> None:
    target.send_cmd(SS_CMD, scmd, bytearray(payload))
    ack_raw = target.simpleserial_wait_ack(timeout=ACK_TIMEOUT_MS)
    ack = ack_to_int(ack_raw)
    if ack is None:
        raise RuntimeError(f'No ACK for cmd=0x{SS_CMD:02x}, scmd=0x{scmd:02x}')
    if ack != 0:
        raise RuntimeError(
            f'Non-zero ACK for cmd=0x{SS_CMD:02x}, scmd=0x{scmd:02x}: raw={ack_raw!r}, int={ack}'
        )


def upload_chunked_vector(name: str, base_scmd: int, values: np.ndarray, chunk_len: int, total_chunks: int) -> None:
    expected_len = chunk_len * total_chunks
    if len(values) != expected_len:
        raise ValueError(f'{name} has len={len(values)} but expected {expected_len}')

    for chunk_index in range(total_chunks):
        start = chunk_index * chunk_len
        end = start + chunk_len
        send_v21_cmd(base_scmd + chunk_index, values[start:end].tobytes())

    print(f'{name} uploaded in {total_chunks} chunks of {chunk_len} bytes')


def upload_reference_vectors() -> None:
    # Limpia cualquier frame residual antes de iniciar la secuencia de chunks.
    target.flush()
    upload_chunked_vector('sk', SCMD_SK_BASE, sk, SK_CHUNK_LEN, SK_CHUNKS)
    upload_chunked_vector('pk', SCMD_PK_BASE, pk, PK_CHUNK_LEN, PK_CHUNKS)
    upload_chunked_vector('ct', SCMD_CT_BASE, ct, CT_CHUNK_LEN, CT_CHUNKS)
    send_v21_cmd(SCMD_SS, ss.tobytes())
    print('ss uploaded in one 32-byte chunk')


print('Prueba 1: LED3 por SimpleSerial')
send_led_command('l', LED3)
send_led_command('h', LED3)
send_led_command('l', LED3)
print('Si LED3 no cambio, no sigas con captura: el problema es comunicacion/programacion.')

Prueba 1: LED3 por SimpleSerial
Si LED3 no cambio, no sigas con captura: el problema es comunicacion/programacion.


In [6]:
print('Prueba 2: cargar vectores por chunks SSv2.1')

# Si estas probando SK/CT personalizados y NO actualizaste `ss`,
# es normal que DECAP devuelva ACK=1 (ss_result != ss_ref).
EXPECT_SS_MATCH = False

# Endurecer la sincronizacion serial antes de enviar chunks.
reset_target(scope)
target.flush()
time.sleep(0.05)
upload_reference_vectors()

if 'ack_to_int' not in globals():
    def ack_to_int(ack) -> int | None:
        if ack is None:
            return None
        if isinstance(ack, int):
            return ack
        if isinstance(ack, (bytes, bytearray)):
            return ack[0] if len(ack) > 0 else 0
        return int(ack)

# --- Verificar que los vectores llegaron correctamente al microcontrolador ---
print('\nVerificando upload: ejecutando DECAP sin scope...')
target.flush()
target.send_cmd(SS_CMD, SCMD_DECAP, bytearray())
ack_raw = target.simpleserial_wait_ack(timeout=READ_TIMEOUT_MS)
ack = ack_to_int(ack_raw)

if ack is None:
    raise RuntimeError('No ACK de DECAP - revisar programacion del micro o comunicacion USB')

if EXPECT_SS_MATCH:
    if ack == 0:
        print(f'PASS: ack_raw={ack_raw!r} -> ack={ack} -> ss_result == ss_ref')
    elif ack == 1:
        raise RuntimeError(
            'ACK=1: ss_result != ss_ref. Si cambiaste sk/ct, debes enviar un ss consistente o usar EXPECT_SS_MATCH=False.'
        )
    else:
        raise RuntimeError(f'ACK no esperado para DECAP: raw={ack_raw!r}, int={ack}')
else:
    if ack in (0, 1):
        meaning = 'match' if ack == 0 else 'mismatch (esperado con sk/ct custom + ss fijo)'
        print(f'OK TEST MODE: ack_raw={ack_raw!r} -> ack={ack} -> {meaning}')
    else:
        raise RuntimeError(f'ACK no esperado para DECAP en modo test: raw={ack_raw!r}, int={ack}')


def capture_once(retries: int = 1) -> np.ndarray:
    last_error = None

    for _ in range(retries):
        try:
            target.flush()
            scope.arm()
            target.send_cmd(SS_CMD, SCMD_DECAP, bytearray())
            ret = scope.capture()
            if ret != 0:
                raise RuntimeError(f'OpenADC timeout/no trigger, ret={ret}')

            ack_raw = target.simpleserial_wait_ack(timeout=READ_TIMEOUT_MS)
            ack = ack_to_int(ack_raw)
            if ack is None:
                raise RuntimeError('Target did not return SSv2.1 ack for decap command')

            if EXPECT_SS_MATCH:
                if ack != 0:
                    raise RuntimeError(f'Target returned non-zero SSv2.1 ack: raw={ack_raw!r}, int={ack}')
            else:
                if ack not in (0, 1):
                    raise RuntimeError(f'Target returned unexpected SSv2.1 ack in test mode: raw={ack_raw!r}, int={ack}')

            return np.asarray(scope.get_last_trace(), dtype=np.float32)
        except Exception as exc:
            last_error = exc
            reset_target(scope)
            target.flush()
            time.sleep(0.1)
            upload_reference_vectors()

    raise RuntimeError(f'Target capture failed after {retries} attempts: {last_error}')


Prueba 2: cargar vectores por chunks SSv2.1


(ChipWhisperer Target WARNING|File SimpleSerial2.py:533) Read timed out: 
(ChipWhisperer Target ERROR|File SimpleSerial2.py:292) Device did not ack


RuntimeError: No ACK for cmd=0x01, scmd=0x00

In [ ]:
# Barrido de llaves SK: sube cada caso, verifica ACK de DECAP y opcionalmente captura una traza.

def run_sk_sweep(sk_cases: dict[str, np.ndarray], *, capture_trace: bool = False) -> pd.DataFrame:
    if 'set_test_vectors' not in globals():
        raise RuntimeError('Primero ejecuta la celda de utilidades de vectores personalizados.')
    if 'upload_reference_vectors' not in globals():
        raise RuntimeError('Primero ejecuta la celda de Prueba 1/2 para definir upload_reference_vectors().')
    if 'ack_to_int' not in globals():
        raise RuntimeError('ack_to_int no esta definido. Ejecuta la celda de Prueba 1/2.')
    if capture_trace and 'capture_once' not in globals():
        raise RuntimeError('capture_once no esta definido. Ejecuta la celda de Prueba 2.')

    rows = []
    sk_original = sk.copy()

    for case_name, sk_case in sk_cases.items():
        sk_case = np.asarray(sk_case, dtype=np.uint8)
        if len(sk_case) != len(sk_original):
            raise ValueError(f'{case_name}: len(sk_case)={len(sk_case)} esperado={len(sk_original)}')

        set_test_vectors(sk_new=sk_case)
        upload_reference_vectors()

        target.flush()
        target.send_cmd(SS_CMD, SCMD_DECAP, bytearray())
        ack_raw = target.simpleserial_wait_ack(timeout=READ_TIMEOUT_MS)
        ack = ack_to_int(ack_raw)

        trace_energy = np.nan
        if capture_trace:
            tr = capture_once(retries=1)
            trace_energy = float(np.mean(np.abs(tr)))

        rows.append({
            'case': case_name,
            'ack_int': ack,
            'ack_raw': repr(ack_raw),
            'sk_first8': ' '.join(f'{int(v):02x}' for v in sk_case[:8]),
            'trace_energy_mean_abs': trace_energy,
        })

    # Restaurar SK original para no dejar estado inesperado.
    set_test_vectors(sk_new=sk_original)

    df = pd.DataFrame(rows)
    display(df)
    return df


# Ejemplo: descomenta para ejecutar un barrido rapido de 4 casos
# sk_cases = {
#     'all_00': np.full_like(sk, 0x00, dtype=np.uint8),
#     'all_ff': np.full_like(sk, 0xFF, dtype=np.uint8),
#     'inc_00_..': (np.arange(len(sk), dtype=np.uint16) & 0xFF).astype(np.uint8),
#     'dec_ff_..': (0xFF - (np.arange(len(sk), dtype=np.uint16) & 0xFF)).astype(np.uint8),
# }
# df_sk = run_sk_sweep(sk_cases, capture_trace=False)
# df_sk

In [ ]:
print('Prueba 3: una sola captura con cmd=0x01 scmd=0x40')
single_trace = capture_once()
plt.figure(figsize=(14, 4))
plt.plot(single_trace)
plt.title('Single capture for isolated cmov trigger')
plt.xlabel('Sample index')
plt.ylabel('Power')
plt.show()

In [ ]:
print('Prueba 4: preview corto')
preview_traces = [single_trace]
for _ in range(PREVIEW_COUNT - 1):
    preview_traces.append(capture_once())

preview_traces = np.vstack(preview_traces)
preview_mean = preview_traces.mean(axis=0)

plt.figure(figsize=(14, 4))
for trace in preview_traces[:5]:
    plt.plot(trace, alpha=0.35)
plt.plot(preview_mean, linewidth=2.0, color='black', label='mean')
plt.title('Preview capture for isolated cmov trigger')
plt.xlabel('Sample index')
plt.ylabel('Power')
plt.legend()
plt.show()

La captura ahora dispara `cmd = 0x01`, `scmd = 0x40` y no espera un paquete de datos de vuelta.

Para este flujo, el comando de decapsulacion solo debe terminar con el `ack` `e` de SimpleSerial v2.1.

In [ ]:
traces = []
capture_rows = []
start_time = time.time()

for trace_index in trange(N_TRACES, desc='Capturing cmov traces'):
    try:
        trace = capture_once()
    except RuntimeError as exc:
        capture_rows.append({
            'trace_index': trace_index,
            'status': 'timeout',
            'samples': 0,
            'error': str(exc),
        })
        continue

    traces.append(trace)
    capture_rows.append({
        'trace_index': trace_index,
        'status': 'ok',
        'samples': len(trace),
        'error': '',
    })

elapsed = time.time() - start_time
capture_log = pd.DataFrame(capture_rows)
good_traces = int((capture_log['status'] == 'ok').sum())
print(f'Captured {good_traces}/{N_TRACES} good traces in {elapsed:.1f}s')

if good_traces == 0:
    raise RuntimeError('No valid traces captured')

trace_matrix = np.vstack(traces)
mean_trace = trace_matrix.mean(axis=0)
std_trace = trace_matrix.std(axis=0)
trace_matrix.shape

In [ ]:
metadata = {
    'platform': PLATFORM,
    'ss_ver': SS_VER,
    'firmware_hex': str(FIRMWARE_HEX),
    'firmware_c': str(FIRMWARE_C),
    'n_traces': int(trace_matrix.shape[0]),
    'n_samples': int(trace_matrix.shape[1]),
    'gain_db': GAIN_DB,
    'adc_offset': ADC_OFFSET,
    'adc_samples': ADC_SAMPLES,
    'ss_cmd': SS_CMD,
    'scmd_map': {
        'sk_base': SCMD_SK_BASE,
        'pk_base': SCMD_PK_BASE,
        'ct_base': SCMD_CT_BASE,
        'ss': SCMD_SS,
        'decap': SCMD_DECAP,
    },
    'chunk_sizes': {
        'sk': SK_CHUNK_LEN,
        'pk': PK_CHUNK_LEN,
        'ct': CT_CHUNK_LEN,
        'ss': len(ss),
    },
    'vector_source': 'firmware_c_arrays',
    'notes': 'cmd=0x01 with SSv2.1 subcommands; trigger isolates PQCLEAN_MLKEM512_CLEAN_cmov',
}

with open(OUTPUT_DIR / 'capture_metadata.json', 'w', encoding='utf-8') as handle:
    json.dump(metadata, handle, indent=2)

print('Guardado:')
print(OUTPUT_DIR / 'traces.npy')
print(OUTPUT_DIR / 'ss_reference.npy')
print(OUTPUT_DIR / 'ss2_reference.npy')
print(OUTPUT_DIR / 'byte_stats.csv')
print(OUTPUT_DIR / 'byte_windows.npy')
print(OUTPUT_DIR / 'capture_metadata.json')

In [ ]:
if 'mean_trace' not in globals() or 'std_trace' not in globals():
    if 'trace_matrix' not in globals():
        raise RuntimeError('Ejecuta primero la celda de captura para definir trace_matrix')
    mean_trace = trace_matrix.mean(axis=0)
    std_trace = trace_matrix.std(axis=0)

plt.figure(figsize=(15, 5))
plt.plot(mean_trace, label='mean trace')
plt.fill_between(
    np.arange(len(mean_trace)),
    mean_trace - std_trace,
    mean_trace + std_trace,
    alpha=0.2,
    label='mean +/- std',
)
plt.title('Averaged cmov trace')
plt.xlabel('Sample index')
plt.ylabel('Power')
plt.legend()
plt.show()

## Rutina automatica por ventanas de byte

Esta parte no usa variacion entre trazas, porque el caso actual repite siempre el mismo material.

La idea es:
- promediar muchas trazas para limpiar ruido
- dividir la region capturada en 32 ventanas, una por iteracion esperada del bucle `cmov`
- extraer una caracteristica simple por ventana
- correlacionar esa caracteristica contra bytes, bits o HW de un vector conocido

Modelo recomendado para `cmov`:
- `delta_ss = ss XOR ss2`

Ese modelo no reemplaza un ataque formal, pero es util para localizar fuga real en el bucle.

In [ ]:
def build_window_features(trace: np.ndarray, byte_count: int = CMOV_BYTES) -> pd.DataFrame:
    trace = np.asarray(trace, dtype=np.float64)
    window_edges = np.linspace(0, len(trace), byte_count + 1, dtype=int)
    rows = []
    for byte_index in range(byte_count):
        start = window_edges[byte_index]
        stop = window_edges[byte_index + 1]
        window = trace[start:stop]
        rows.append({
            'byte_index': byte_index,
            'start': start,
            'stop': stop,
            'mean': float(window.mean()),
            'abs_mean': float(np.mean(np.abs(window))),
            'energy': float(np.sum(window ** 2)),
            'ptp': float(np.ptp(window)),
        })
    return pd.DataFrame(rows)


def correlation_score(feature: np.ndarray, model: np.ndarray) -> float:
    feature = np.asarray(feature, dtype=np.float64)
    model = np.asarray(model, dtype=np.float64)
    if feature.std() == 0 or model.std() == 0:
        return 0.0
    return float(np.corrcoef(feature, model)[0, 1])


def build_model_table(model_name: str = DEFAULT_MODEL_NAME) -> pd.DataFrame:
    model_bytes = np.asarray(reference_vectors[model_name][:CMOV_BYTES], dtype=np.uint8)
    table = pd.DataFrame({
        'byte_index': np.arange(CMOV_BYTES),
        'value': model_bytes,
        'hw': hw_u8(model_bytes),
    })
    for bit_index in range(8):
        table[f'bit_{bit_index}'] = bit_column(model_bytes, bit_index)
    return table


def rank_models(mean_trace: np.ndarray, model_name: str = DEFAULT_MODEL_NAME) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    feature_table = build_window_features(mean_trace, CMOV_BYTES)
    model_table = build_model_table(model_name)
    merged = feature_table.merge(model_table, on='byte_index', how='inner')

    feature_columns = ['mean', 'abs_mean', 'energy', 'ptp']
    model_columns = ['value', 'hw'] + [f'bit_{bit_index}' for bit_index in range(8)]
    rows = []
    for feature_column in feature_columns:
        feature_vector = merged[feature_column].to_numpy()
        for model_column in model_columns:
            score = correlation_score(feature_vector, merged[model_column].to_numpy())
            rows.append({
                'feature': feature_column,
                'model': model_column,
                'corr': score,
                'abs_corr': abs(score),
            })
    ranking = pd.DataFrame(rows).sort_values('abs_corr', ascending=False).reset_index(drop=True)
    return merged, ranking, model_table

In [ ]:
MODEL_NAME = CT_FIRST32 #SK_FIRST32
window_table, ranking_table, model_table = rank_models(mean_trace, MODEL_NAME)

display(ranking_table.head(12))
display(window_table.head())

In [ ]:
best_feature = ranking_table.iloc[0]['feature']
best_model = ranking_table.iloc[0]['model']

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].plot(window_table['byte_index'], window_table[best_feature], marker='o')
axes[0].set_title(f'Window feature: {best_feature}')
axes[0].set_ylabel('Feature value')

axes[1].plot(window_table['byte_index'], window_table[best_model], marker='o', color='tab:red')
axes[1].set_title(f'Model: {MODEL_NAME}.{best_model}')
axes[1].set_xlabel('Byte index in cmov loop')
axes[1].set_ylabel('Model value')
plt.tight_layout()
plt.show()

run_tag = f'{PLATFORM.lower()}_{MODEL_NAME}'

ranking_csv_path = OUTPUT_DIR / f'cmov_window_ranking_{run_tag}.csv'
window_csv_path = OUTPUT_DIR / f'cmov_window_features_{run_tag}.csv'
ranking_table.to_csv(ranking_csv_path, index=False)
window_table.to_csv(window_csv_path, index=False)
print(ranking_csv_path)
print(window_csv_path)

## Pruebas rapidas de modelos alternos

Cambia `MODEL_NAME` por uno de estos valores y vuelve a ejecutar las dos celdas anteriores:
- `delta_ss`
- `ss`
- `ss2`
- `sk_first32`
- `ct_first32`

In [ ]:
scope.dis()
target.dis()
print('Scope and target disconnected')